# Phase 1 — Data Collection
Runs the data_collection modules and displays results. All logic lives in `src/data_collection/`.

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))
import pandas as pd

from src.data_collection import config
from src.data_collection.downloader import fetch_adjusted_close, fetch_index_prices
from src.data_collection.validator import clean_prices, validate_prices
from src.data_collection.returns import calculate_log_returns
from src.data_collection.saver import save_raw, save_processedfrom src.data_collection.sectors import get_sector_map


In [ ]:
# Collect
raw_prices = fetch_adjusted_close(config.TICKERS, config.START_DATE, config.END_DATE)
index_prices = fetch_index_prices(config.INDEX_TICKER, config.START_DATE, config.END_DATE)
save_raw(raw_prices, 'prices_adjusted_close_raw.csv', config.DATA_RAW_DIR)
save_raw(index_prices, 'index_prices_sp500_raw.csv', config.DATA_RAW_DIR)

In [ ]:
# Clean + derive returns
clean_price_matrix, cleaning_report = clean_prices(raw_prices)
returns_matrix = calculate_log_returns(clean_price_matrix)
index_prices_aligned = index_prices.reindex(clean_price_matrix.index).ffill()

save_processed(clean_price_matrix, 'prices_adjusted_close.csv', config.DATA_PROCESSED_DIR)
save_processed(returns_matrix, 'returns_daily_log.csv', config.DATA_PROCESSED_DIR)
save_processed(index_prices_aligned, 'index_prices_sp500.csv', config.DATA_PROCESSED_DIR)

print(cleaning_report)

In [ ]:
# Validate
result = validate_prices(clean_price_matrix, returns_matrix)
result

In [ ]:
# Sector lookup (per teammate review comment)
sector_map = get_sector_map(config.TICKERS)
sector_df = pd.DataFrame(list(sector_map.items()), columns=['ticker', 'sector'])
save_processed(sector_df.set_index('ticker'), 'sector_map.csv', config.DATA_PROCESSED_DIR)
sector_df

In [ ]:
clean_price_matrix.tail()

In [ ]:
returns_matrix.tail()